In [ ]:
import math
import torch
import gpytorch
from matplotlib import pyplot as plt
import h5py
import numpy as np
import random
import bacco

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import emcee
import corner

In [ ]:
import os
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/src")
import utils

In [ ]:
import sys
sys.path.insert(0, "/cosmos_storage/home/fgmaion/MTNG-resims/scripts/train")
from GP_models import BHMF_Model

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

### Get Parameters

In [ ]:
wind_en_or      = []
wind_vel_or     = []
rho_rec_or      = []
sf_ts_or        = []
ef_kin_or       = []
ef_high_or      = []
f_re_or         = []

for i in range(31):
    if i<30:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro_{:d}.txt".format(i)
    else:
        filename = "/cosmos_storage/simulations/TNG_Family/MN5_resims/param_LH/param_MTNG-hydro.txt"

    with open(filename, 'r') as f:
        for line in f.readlines():
            if len(line.split())!=0:
                if line.split()[0] == 'WindEnergyIn1e51erg':
                    wind_en_or.append(float(line.split()[1]))
                if line.split()[0] == 'VariableWindVelFactor':
                    wind_vel_or.append(float(line.split()[1]))
                if line.split()[0] == 'WindFreeTravelDensFac':
                    rho_rec_or.append(float(line.split()[1]))
                if line.split()[0] == 'MaxSfrTimescale':
                    sf_ts_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackFactor':
                    ef_kin_or.append(float(line.split()[1]))
                if line.split()[0] == 'BlackHoleFeedbackFactor':
                    ef_high_or.append(float(line.split()[1]))
                if line.split()[0] == 'RadioFeedbackReiorientationFactor':
                    f_re_or.append(float(line.split()[1]))

rho_rec_or = np.log10(rho_rec_or)
ef_kin_or = np.log10(ef_kin_or)
        
wind_en   = (np.asarray(wind_en_or) - np.mean(wind_en_or)) / np.std(wind_en_or)
wind_vel  = (np.asarray(wind_vel_or) - np.mean(wind_vel_or)) / np.std(wind_vel_or)
rho_rec   = (np.asarray(rho_rec_or) - np.mean(rho_rec_or)) / np.std(rho_rec_or)
sf_ts     = (np.asarray(sf_ts_or) - np.mean(sf_ts_or)) / np.std(sf_ts_or)
ef_kin    = (np.asarray(ef_kin_or) - np.mean(ef_kin_or)) / np.std(ef_kin_or)
ef_high   = (np.asarray(ef_high_or) - np.mean(ef_high_or)) / np.std(ef_high_or)
f_re      = (np.asarray(f_re_or) - np.mean(f_re_or)) / np.std(f_re_or)

In [ ]:
name_list = ['LH_{:d}'.format(i) for i in range(30)] + ['fiducial']

In [ ]:
fid_pars = np.array([wind_en[30], wind_vel[30], rho_rec[30], sf_ts[30], ef_kin[30], ef_high[30], f_re[30]])

weights = (0.1, 1, 0.1, 1, 0.1, 0.1, 0.1)

dist = np.zeros(31)
for i in range(len(name_list)):
    pars_i = np.array([wind_en[i], wind_vel[i], rho_rec[i], sf_ts[i], ef_kin[i], ef_high[i], f_re[i]])
    dist[i] = np.sum((weights*(pars_i - fid_pars))**2)

In [ ]:
def pars(i, mbh):

    arr = np.vstack( ( mbh, np.ones(len(mbh)) * wind_en[i],\
                        np.ones(len(mbh)) * wind_vel[i],\
                        np.ones(len(mbh)) * rho_rec[i],\
                        np.ones(len(mbh)) * sf_ts[i],\
                        np.ones(len(mbh)) * ef_kin[i],\
                        np.ones(len(mbh)) * ef_high[i],\
                        np.ones(len(mbh)) * f_re[i])).T

    return arr

In [ ]:
Nbins_bhmf = 10

zoom_bhmf = {}
for i in range(len(name_list)):
    zoom_bhmf[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/bhmf/bhmf_{}_Nbins{:d}.npy".format(name_list[i], Nbins_bhmf), allow_pickle=True)[0]

In [ ]:
# Filter NaNs
for i in range(len(name_list)):
    mask = ~np.isnan(zoom_bhmf[name_list[i]]['bhmf'][0]) & ~np.isnan(zoom_bhmf[name_list[i]]['mbh'][0])
    
    zoom_bhmf[name_list[i]]['bhmf'][0] = zoom_bhmf[name_list[i]]['bhmf'][0][mask]
    zoom_bhmf[name_list[i]]['mbh'][0] = zoom_bhmf[name_list[i]]['mbh'][0][mask]

In [ ]:
train_sel = np.argsort(dist)[::-1][:25]
test_sel = list(set(range(31)) - set(train_sel))

In [ ]:
#del bhmf_global, arr_global
pars_global = pars(train_sel[0], np.log10(zoom_bhmf[name_list[train_sel[0]]]['mbh']) )
bhmf_global = np.log10(zoom_bhmf[name_list[train_sel[0]]]['bhmf'])

for i in range(len(train_sel)):
    mbh = np.log10(zoom_bhmf[name_list[train_sel[i]]]['mbh'])

    arr = pars(train_sel[i], mbh)

    pars_global = np.vstack((pars_global, arr))

    bhmf_global = np.hstack((bhmf_global, np.log10(zoom_bhmf[name_list[train_sel[i]]]['bhmf'])))

In [ ]:
train_x = torch.asarray(pars_global, dtype=torch.float)
train_y = torch.asarray(bhmf_global, dtype=torch.float)

In [ ]:
# initialize likelihood and model
likelihood = gpytorch.likelihoods.GaussianLikelihood(noise_constraint=gpytorch.constraints.GreaterThan(1e-2))
model = BHMF_Model(train_x, train_y, likelihood)

In [ ]:
torch.set_num_threads(8)

#### Train it

In [ ]:
# this is for running the notebook in our testing framework
import os
import copy
smoke_test = ('CI' in os.environ)
#training_iter = 2 if smoke_test else 200

n_restarts = 30
steps_per_restart = 30

best_state = None
best_loss = float('inf')

for r in range(n_restarts):
    print(f"\n=== Restart {r+1}/{n_restarts} ===")
    model.initialize()

    # Find optimal model hyperparameters
    model.train()
    likelihood.train()

    # Use the adam optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # Includes GaussianLikelihood parameters

    # "Loss" for GPs - the marginal log likelihood
    mll = gpytorch.mlls.ExactMarginalLogLikelihood(likelihood, model)

    for i in range(steps_per_restart):
        # Zero gradients from previous iteration
        optimizer.zero_grad()
        # Output from model
        output = model(train_x)
        # Calc loss and backprop gradients
        loss = -torch.sum(mll(output, train_y))
        loss.backward()
        optimizer.step()

    # compute final loss for this restart
    with torch.no_grad():
        final_output = model(train_x)
        final_loss = float(-mll(final_output, train_y))

    print(f"Final loss restart {r+1}: {final_loss:.4f}")

    # store best
    if final_loss < best_loss:
        best_loss = final_loss
        best_state = {
            'model': copy.deepcopy(model.state_dict()),
            'likelihood': copy.deepcopy(likelihood.state_dict()),
        }

In [ ]:
# ---- restore best ----
model.load_state_dict(best_state['model'])
likelihood.load_state_dict(best_state['likelihood'])

In [ ]:
print(model.covar_module.base_kernel.lengthscale)
print(model.covar_module.outputscale)
print(torch.sqrt(model.likelihood.noise_covar.noise))

In [ ]:
# Get into evaluation (predictive posterior) mode
model.eval()
likelihood.eval()

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(5, 5, figsize=(30, 30), dpi=150)

    for i in range(5):
        for j in range(5):
            test_x = torch.asarray(pars(train_sel[5*i+j], np.log10(zoom_bhmf[name_list[train_sel[5*i+j]]]['mbh'])), dtype=torch.float)
            observed_pred = likelihood(model(torch.asarray(test_x, dtype=torch.float)))

            ax[i,j].set_title('Simulation {:s}'.format(name_list[train_sel[5*i+j]]), fontsize=16)

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(\Phi/[M_{\odot}\mathrm{Mpc}^{-3}])$', fontsize=16)
            if i==4:
                ax[i,j].set_xlabel('$M_*[M_{\odot}]$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()
            # Plot training data as black stars
            ax[i,j].plot(np.log10(zoom_bhmf[name_list[train_sel[5*i+j]]]['mbh']), np.log10(zoom_bhmf[name_list[train_sel[5*i+j]]]['bhmf']), 'b*')

            # Plot predictive means as blue line
        #    for i in range(10):
        #        ax.plot(mbh_test, bhmf_test.sample().numpy(), 'b', ls='--', lw=0.5)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), 'b')

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='b', alpha=0.5, edgecolor=None)

ax[0,0].legend(loc='lower left', fontsize=18)

In [ ]:
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    # Initialize plot
    f, ax = plt.subplots(2, 3, figsize=(15, 10), dpi=100)
    plt.subplots_adjust(wspace=0.15, hspace=0.3)

    for i in range(2):
        for j in range(3):
            test_x = torch.asarray(pars(test_sel[3*i+j], np.log10(zoom_bhmf[name_list[test_sel[3*i+j]]]['mbh'])), dtype=torch.float)
            observed_pred = likelihood(model(torch.asarray(test_x, dtype=torch.float)))

            ax[i,j].set_title('Simulation {:s}'.format(name_list[test_sel[3*i+j]]), fontsize=16)

            if j==0:
                ax[i,j].set_ylabel('$\log_{10}(\Phi/[\mathrm{Mpc}^{-3}\mathrm{dex}^{-1}])$', fontsize=16)
            
            ax[i,j].set_xlabel('$\log_{10}(M_\mathrm{BH}/M_{\odot})$', fontsize=16)

            # Get upper and lower confidence bounds
            lower, upper = observed_pred.confidence_region()

            # Shade between the lower and upper confidence bounds
            ax[i,j].fill_between(test_x[:,0].numpy(), observed_pred.mean.numpy()-observed_pred.stddev.numpy(), observed_pred.mean.numpy()+observed_pred.stddev.numpy(), color='C0', alpha=0.4, edgecolor=None)
            ax[i,j].plot(test_x[:,0], observed_pred.mean.numpy(), color='#0077BB', lw=2, label="GP Prediction", alpha=0.9)

            # Plot test data as blue squares
            ax[i,j].plot(np.log10(zoom_bhmf[name_list[test_sel[3*i+j]]]['mbh']), np.log10(zoom_bhmf[name_list[test_sel[3*i+j]]]['bhmf']), 's', color='#0077BB', label="Test Data")

ax[0,0].legend(loc='lower left', fontsize=12)

plt.savefig("/cosmos_storage/home/fgmaion/MTNG-resims/results/testing_plots/bhmf_test.pdf", bbox_inches='tight')

### Now look at model trained with full data

In [ ]:
os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/scripts")
from GP_models import bhmf_Model, fgas_Model

In [ ]:
model_bhmf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_model_bhmf.pth")
likelihood_bhmf = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_likelihood_bhmf.pth")

model_bhmf.eval()
likelihood_bhmf.eval()

In [ ]:
print(model_bhmf.covar_module.base_kernel.lengthscale)
print(model_bhmf.covar_module.outputscale)
print(torch.sqrt(model_bhmf.likelihood.noise_covar.noise))

In [ ]:
def get_theta_1p_vars(param_num, mbh, nvars):
    fid_theta = np.array([wind_en[30], wind_vel[30], rho_rec[30], sf_ts[30], ef_kin[30], ef_high[30], f_re[30]])

    theta_1p_vars = {}

    delta = 2 / nvars
    for i in range(nvars+1):
        par_array = np.copy(fid_theta)
        par_array[param_num] += (i - nvars//2) * delta
        theta_1p_vars[i] = np.vstack( ( mbh, np.ones(len(mbh)) * par_array[0],\
                                        np.ones(len(mbh)) * par_array[1],\
                                        np.ones(len(mbh)) * par_array[2],\
                                        np.ones(len(mbh)) * par_array[3],\
                                        np.ones(len(mbh)) * par_array[4],\
                                        np.ones(len(mbh)) * par_array[5],\
                                        np.ones(len(mbh)) * par_array[6])).T

    return theta_1p_vars

In [ ]:
fig, ax = plt.subplots(2, 4, dpi=200, sharey=False, figsize=(15,7))

# Paul Tol's discrete rainbow scheme - selecting 4 well-separated colors
# for the -2Δ, -Δ, +Δ, +2Δ variations (Fiducial stays black)
tol_rainbow = ['#3F60AE', '#6DB388', '#E68B33', '#D33C2A']  # blue, green, orange, red

mbh_test = np.linspace(9,12.5,50)
names = ['Wind Energy', 'Wind Velocity', 'Density for Recoupling', 'Max SFR Timescale', 'AGN Kinetic Feedback', 'AGN High Accretion Feedback', 'AGN Reorientation Factor']

for ax_i in np.ndarray.flatten(ax):
    ax_i.set_ylim(-0.4,0.4)
    # Thicker plot edges
    for spine in ax_i.spines.values():
        spine.set_linewidth(2.5)

    # Major and minor ticks
    ax_i.tick_params(axis='both', which='major', width=2.5, length=8, labelsize=12, direction='in', right=True, top=True)
    ax_i.tick_params(axis='both', which='minor', width=1.5, length=4, direction='in', right=True, top=True)
    ax_i.minorticks_on()

labels=['$-2\Delta$', '$-\Delta$', 'Fiducial', '$+\Delta$', '$+2\Delta$']

# Map n=0,1,3,4 to tol_rainbow indices 0,1,2,3
color_map = {0: tol_rainbow[0], 1: tol_rainbow[1], 3: tol_rainbow[2], 4: tol_rainbow[3]}

for i in range(2):
    for j in range(4):
        if 2*j+i >= 7:
            ax[i,j].set_xticks([])
            ax[i,j].set_yticks([])
            for spine in ax[i,j].spines.values():
                spine.set_visible(False)
            ax[i,j].minorticks_off()
            continue

        ax[i,j].set_title(names[2*j+i])
        ax[i,j].set_xlabel(r"$\log_{10}[M_*/M_\odot]$", fontsize=16)
        theta_vars = get_theta_1p_vars(2*j+i, mbh_test, 5)

        mu_fid, var_fid = model_bhmf(torch.asarray(theta_vars[2], dtype=torch.float)).mean.detach().numpy(), model_bhmf(torch.asarray(theta_vars[2], dtype=torch.float)).variance.detach().numpy()

        for n in range(5):
            mu_pred, var = model_bhmf(torch.asarray(theta_vars[n], dtype=torch.float)).mean.detach().numpy(), model_bhmf(torch.asarray(theta_vars[n], dtype=torch.float)).variance.detach().numpy()
            if n == 2:
                ax[i,j].plot(mbh_test, mu_pred - mu_fid, label="Fiducial", ls='--', color='k', lw=3)
            else:
                ax[i,j].plot(mbh_test, mu_pred - mu_fid, color=color_map[n], label=labels[n], lw=3)

for i in range(3):
    ax[0,i+1].set_yticklabels([])
    ax[1,i+1].set_yticklabels([])

# Y-axis labels on the leftmost column
ax[0,0].set_ylabel(r"$\Delta \log_{10}\Phi$", fontsize=16)
ax[1,0].set_ylabel(r"$\Delta \log_{10}\Phi$", fontsize=16)

# Legend in the empty panel
handles, leg_labels = ax[0,0].get_legend_handles_labels()
order = [0, 1, 2, 3, 4]
handles = [handles[k] for k in order]
leg_labels = [leg_labels[k] for k in order]

ax[1,3].legend(handles, leg_labels, loc='center', fontsize=14, frameon=False, title='Parameter Variation', title_fontsize=15)

plt.subplots_adjust(wspace=0.0, hspace=0.5)